In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [6]:
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()


In [ ]:
model = Pipeline([
    ("scalar", StandardScaler()), 
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

In [ ]:
logistic_regression = model["model"] # extract trained model
scalar = model["scalar"] # get scalar used by model

X_train_scaled = pd.DataFrame(scalar.transform(X_train), columns=X_train.columns) # transform train and test data
X_test_scaled = pd.DataFrame(scalar.transform(X_test), columns=X_test.columns)


masker = shap.maskers.Independent(X_train_scaled) # background dataset to simulate missing features, assumes features are independent and replaces with values sampled from X_train
explainer = shap.LinearExplainer(logistic_regression, masker=masker) # compute SHAP values

shap_values = explainer.shap_values(X_train_scaled) # compute contribution from each feature in classification

print(f"SHAP values shape: {shap_values.shape}") # match X_train shape?